<a href="https://colab.research.google.com/github/RiddhikaJayashree/GEN-AI-AND-LLM/blob/main/ex5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
# ---------- Sentiment Analysis ---------
sentiment_analyzer = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")
reviews = [
    "The new smartphone has an amazing camera and battery life!",
    "The delivery was late and the packaging was damaged."
]
for review in reviews:
    result = sentiment_analyzer(review)[0]  # Grab the first item from the list
    print(f"Review: {review}\n-> {result['label']} ({round(result['score'], 3)})\n")
# ---------- Document Classification (Zero-Shot) ---------
# Bypass the pipeline completely to avoid label formatting limits
model_name = "facebook/bart-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
document = "The central bank raised interest rates to control rising inflation."
candidate_labels = ["Politics", "Economy", "Sports", "Technology"]
# Format inputs for natural language inference (premise vs hypothesis)
results = []
for label in candidate_labels:
    hypothesis = f"This text is about {label}."
    inputs = tokenizer(document, hypothesis, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)
    # The model predicts entailment (index 2), neutral (index 1), and contradiction (index 0)
    # We apply softmax over the contradiction vs entailment logits to get a confidence score
    logits = outputs.logits[0]
    entail_contradict_logits = logits[[0, 2]]
    probs = torch.softmax(entail_contradict_logits, dim=0)
    entail_prob = probs[1].item()
    results.append((label, entail_prob))
# Sort labels by score in descending order
results.sort(key=lambda x: x[1], reverse=True)
print("Document:", document)
for label, score in results:
    print(f"{label}: {round(score, 3)}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Review: The new smartphone has an amazing camera and battery life!
-> POSITIVE (1.0)

Review: The delivery was late and the packaging was damaged.
-> NEGATIVE (1.0)



Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Document: The central bank raised interest rates to control rising inflation.
Economy: 0.655
Politics: 0.06
Technology: 0.026
Sports: 0.004
